## Producer — poll CoinGecko, send to Event Hub (Kafka protocol)

In [0]:
dbutils.widgets.text("eventhub_namespace", "evhua5816bd")
dbutils.widgets.text("eventhub_name", "crypto-ticks")
dbutils.widgets.text("secret_scope", "team-crypto-scope")
dbutils.widgets.text("symbols", "bitcoin,ethereum,solana")
dbutils.widgets.text("poll_seconds", "10")
dbutils.widgets.text("iterations", "30")


In [0]:
import json
import time
import requests
from kafka import KafkaProducer

eventhub_namespace = dbutils.widgets.get("eventhub_namespace")
eventhub_name = dbutils.widgets.get("eventhub_name")
secret_scope = dbutils.widgets.get("secret_scope")
symbols = dbutils.widgets.get("symbols").split(",")
poll_seconds = int(dbutils.widgets.get("poll_seconds"))
iterations = int(dbutils.widgets.get("iterations"))

connection_str = dbutils.secrets.get(scope=secret_scope, key="eventhub-connection-string")


In [0]:
producer = KafkaProducer(
    bootstrap_servers=f"{eventhub_namespace}.servicebus.windows.net:9093",
    security_protocol="SASL_SSL",
    sasl_mechanism="PLAIN",
    sasl_plain_username="$ConnectionString",
    sasl_plain_password=connection_str,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)


In [0]:
ids = ",".join(symbols)

for _ in range(iterations):
    resp = requests.get(
        "https://api.coingecko.com/api/v3/simple/price",
        params={"ids": ids, "vs_currencies": "usd"},
    )
    data = resp.json()
    now = time.time()
    for symbol in symbols:
        if symbol in data:
            event = {"symbol": symbol, "price_usd": data[symbol]["usd"], "ts": now}
            producer.send(eventhub_name, value=event)
    producer.flush()
    time.sleep(poll_seconds)
